In [11]:
import numpy as np
import nltk
import re
from scipy import spatial
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords
import networkx as nx

In [12]:
nltk.download('omw-1.4')
import ssl
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\morga\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\morga\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\morga\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\morga\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [13]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip -d data/

'wget' is not recognized as an internal or external command,
operable program or batch file.
'unzip' is not recognized as an internal or external command,
operable program or batch file.


In [26]:
def loadGloveModel(file):
    gloveModel = {}
    with open(file,"r",encoding="utf-8") as f:
        for line in f:
            values = line.split()
            word = values[0]
            array = np.asarray(values[1:],dtype="float32")
            gloveModel[word] = array

    print(len(gloveModel))
    return gloveModel

model = loadGloveModel(r"C:\Users\morga\OneDrive\Desktop\Jetlearn\Deep learning\glove.6B\glove.6B.100d.txt")

400000


In [16]:
text = """Mary had a little lamb,  little lamb, little lamb,  Mary had a little lamb, its fleece was white as snow. And everywhere that Mary went,  Mary went, Mary went,  and everywhere that Mary went, the lamb was sure to go. It followed her to school one day  school one day, school one day,  It followed her to school one day, which was against the rules. It made the children laugh and play,  laugh and play, laugh and play,  it made the children laugh and play to see And so the teacher turned it out,  turned it out, turned it out,  And so the teacher turned it out, but still it lingered near, And waited patiently about,  patiently about, patiently about,  And waited patiently about till Mary did appear. "Why does the lamb love Mary so?"  Love Mary so? Love Mary so?  "Why does the lamb love Mary so," the eager children cry. "Why, Mary loves the lamb, you know."  The lamb, you know, the lamb, you know, "Why, Mary loves the lamb, you know," the teacher did repl"""
sentences = sent_tokenize(text)
print(len(sentences))
sentNew = [re.sub(r"[^\w\s]","",s.lower()) for s in sentences] #Not: letters, number, underscore, space
print(sentNew)
stop_words = stopwords.words("english")
sentTokens = [[w for w in s.split() if w not in stop_words] for s in sentNew]
print(sentTokens)

10
['mary had a little lamb  little lamb little lamb  mary had a little lamb its fleece was white as snow', 'and everywhere that mary went  mary went mary went  and everywhere that mary went the lamb was sure to go', 'it followed her to school one day  school one day school one day  it followed her to school one day which was against the rules', 'it made the children laugh and play  laugh and play laugh and play  it made the children laugh and play to see and so the teacher turned it out  turned it out turned it out  and so the teacher turned it out but still it lingered near and waited patiently about  patiently about patiently about  and waited patiently about till mary did appear', 'why does the lamb love mary so', 'love mary so', 'love mary so', 'why does the lamb love mary so the eager children cry', 'why mary loves the lamb you know', 'the lamb you know the lamb you know why mary loves the lamb you know the teacher did repl']
[['mary', 'little', 'lamb', 'little', 'lamb', 'little'

In [27]:
def getEmbed(token,model):
    embedding = []
    for word in token:
        if word in model:
            embedding.append(model[word])

    if embedding:
        return np.mean(embedding,axis=0)
    else:
        return np.zeros(100)

In [28]:
sentEmbed = [getEmbed(sentence,model) for sentence in sentTokens]

In [29]:
simMatrix = np.zeros((len(sentEmbed),len(sentEmbed)))
for i,rowEmbed in enumerate(sentEmbed):
    for j, colEmbed in enumerate(sentEmbed):
        simMatrix[i][j] = 1-spatial.distance.cosine(rowEmbed,colEmbed)
graph = nx.from_numpy_array(simMatrix)
ranks = nx.pagerank(graph)
sentScores = {sent:ranks[index] for index,sent in enumerate(sentences)}
top4 = sorted(sentScores.items(),key=lambda x:x[1],reverse=True)[0:4]
for sent in top4:
    print(sent[0])

"Why does the lamb love Mary so," the eager children cry.
"Why, Mary loves the lamb, you know."
"Why does the lamb love Mary so?"
And everywhere that Mary went,  Mary went, Mary went,  and everywhere that Mary went, the lamb was sure to go.
